# Analyse des vulnérabilités ANSSI

Ce notebook charge le fichier CSV généré par le pipeline, explore les données, produit un ensemble de visualisations et applique des modèles de Machine Learning.

## 1. Chargement des données

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.dates as mdates
import seaborn as sns
from sklearn.cluster import KMeans
from sklearn.preprocessing import LabelEncoder, StandardScaler
from sklearn.ensemble import RandomForestClassifier
from sklearn.model_selection import train_test_split
from sklearn.metrics import classification_report, confusion_matrix, silhouette_score
from sklearn.decomposition import PCA

sns.set_theme(style='whitegrid')
plt.rcParams['figure.dpi'] = 100

df = pd.read_csv('donnees_enrichies.csv')

df['Score CVSS'] = pd.to_numeric(df['Score CVSS'], errors='coerce')
df['Score EPSS'] = pd.to_numeric(df['Score EPSS'], errors='coerce')
df['Date de publication'] = pd.to_datetime(df['Date de publication'], errors='coerce')

# Sous-ensembles propres
df_cvss = df.dropna(subset=['Score CVSS'])
df_epss = df.dropna(subset=['Score EPSS'])
df_deux = df.dropna(subset=['Score CVSS', 'Score EPSS'])
df_sev = df[df['Base Severity'].isin(['LOW','MEDIUM','HIGH','CRITICAL'])]

print(f'Lignes totales : {len(df)}')
print(f'Colonnes : {list(df.columns)}')
print(f'CVE avec CVSS : {len(df_cvss)}')
print(f'CVE avec EPSS : {len(df_epss)}')

## 2. Exploration du DataFrame

In [ ]:
df.head(10)

In [ ]:
df.describe()

In [ ]:
print('Valeurs manquantes par colonne :')
print(df.isnull().sum())
print()
print('Répartition Type de bulletin :')
print(df['Type'].value_counts())
print()
print('Répartition Sévérité :')
print(df['Base Severity'].value_counts())

## 3. Visualisations

### 3.1 Distribution des scores CVSS

In [ ]:
fig, ax = plt.subplots(figsize=(10, 5))
ax.hist(df_cvss['Score CVSS'], bins=20, color='steelblue', edgecolor='white')
for x, label in [(3.9,'Low'), (6.9,'Medium'), (8.9,'High')]:
    ax.axvline(x, color='red', linestyle='--', linewidth=0.8)
    ax.text(x+0.05, ax.get_ylim()[1]*0.9, label, color='red', fontsize=8)
ax.set_title('Distribution des scores CVSS')
ax.set_xlabel('Score CVSS')
ax.set_ylabel('Nombre de CVE')
plt.tight_layout()
plt.savefig('graphique_cvss.png')
plt.show()

### 3.2 Distribution des scores EPSS

In [ ]:
fig, ax = plt.subplots(figsize=(10, 5))
ax.hist(df_epss['Score EPSS'], bins=30, color='coral', edgecolor='white')
ax.axvline(0.1, color='orange', linestyle='--', linewidth=0.8, label='Seuil 0.1')
ax.axvline(0.3, color='red', linestyle='--', linewidth=0.8, label='Seuil 0.3')
ax.set_title('Distribution des scores EPSS')
ax.set_xlabel('Score EPSS')
ax.set_ylabel('Nombre de CVE')
ax.legend()
plt.tight_layout()
plt.savefig('graphique_epss.png')
plt.show()
print(f"CVE avec EPSS > 0.3 : {(df_epss['Score EPSS'] > 0.3).sum()}")

### 3.3 Répartition des niveaux de sévérité

In [ ]:
couleurs_sev = {'LOW':'#2196F3','MEDIUM':'#FF9800','HIGH':'#F44336','CRITICAL':'#9C27B0'}
compte = df_sev['Base Severity'].value_counts()
ordre = [s for s in ['CRITICAL','HIGH','MEDIUM','LOW'] if s in compte.index]
compte = compte[ordre]
fig, ax = plt.subplots(figsize=(8, 8))
ax.pie(compte, labels=compte.index, autopct='%1.1f%%', startangle=140,
       colors=[couleurs_sev[s] for s in ordre])
ax.set_title('Répartition des niveaux de sévérité')
plt.tight_layout()
plt.savefig('graphique_severite.png')
plt.show()

### 3.4 Top 10 des éditeurs les plus touchés

In [ ]:
top_vendors = df['Éditeur (Vendor)'].replace('n/a', pd.NA).dropna().value_counts().head(10)
fig, ax = plt.subplots(figsize=(10, 5))
top_vendors.plot(kind='bar', ax=ax, color='mediumseagreen', edgecolor='white')
ax.set_title('Top 10 des éditeurs les plus touchés')
ax.set_xlabel('Éditeur')
ax.set_ylabel('Nombre de CVE')
ax.tick_params(axis='x', rotation=45)
plt.tight_layout()
plt.savefig('graphique_vendors.png')
plt.show()

### 3.5 Top 10 des produits les plus touchés

In [ ]:
top_produits = df['Produit'].replace('n/a', pd.NA).dropna().value_counts().head(10)
fig, ax = plt.subplots(figsize=(10, 5))
top_produits.plot(kind='bar', ax=ax, color='mediumpurple', edgecolor='white')
ax.set_title('Top 10 des produits les plus touchés')
ax.set_xlabel('Produit')
ax.set_ylabel('Nombre de CVE')
ax.tick_params(axis='x', rotation=45)
plt.tight_layout()
plt.savefig('graphique_produits.png')
plt.show()

### 3.6 Nuage de points : Score CVSS vs Score EPSS

In [ ]:
couleurs_map = {'CRITICAL':'#9C27B0','HIGH':'#F44336','MEDIUM':'#FF9800','LOW':'#2196F3'}
df_plot = df_deux[df_deux['Base Severity'].isin(couleurs_map)].copy()
fig, ax = plt.subplots(figsize=(10, 6))
for sev, grp in df_plot.groupby('Base Severity'):
    ax.scatter(grp['Score CVSS'], grp['Score EPSS'], label=sev,
               color=couleurs_map[sev], alpha=0.7, edgecolors='white', linewidth=0.3)
ax.set_title('Score CVSS vs Score EPSS (coloré par sévérité)')
ax.set_xlabel('Score CVSS')
ax.set_ylabel('Score EPSS')
ax.legend(title='Sévérité')
plt.tight_layout()
plt.savefig('graphique_cvss_vs_epss.png')
plt.show()

### 3.7 Heatmap des corrélations (CVSS / EPSS)

In [ ]:
corr = df_deux[['Score CVSS', 'Score EPSS']].corr()
fig, ax = plt.subplots(figsize=(6, 4))
sns.heatmap(corr, annot=True, fmt='.2f', cmap='coolwarm', ax=ax,
            linewidths=0.5, square=True, vmin=-1, vmax=1)
ax.set_title('Heatmap des corrélations CVSS / EPSS')
plt.tight_layout()
plt.savefig('graphique_heatmap.png')
plt.show()
print(f"Corrélation CVSS-EPSS : {corr.loc['Score CVSS','Score EPSS']:.3f}")

### 3.8 Boxplot des scores CVSS par éditeur (top 8)

In [ ]:
top8 = df_cvss['Éditeur (Vendor)'].replace('n/a', pd.NA).dropna().value_counts().head(8).index
df_box = df_cvss[df_cvss['Éditeur (Vendor)'].isin(top8)]
fig, ax = plt.subplots(figsize=(12, 6))
order = (df_box.groupby('Éditeur (Vendor)')['Score CVSS']
         .median().sort_values(ascending=False).index)
sns.boxplot(data=df_box, x='Éditeur (Vendor)', y='Score CVSS',
            order=order, palette='Set2', ax=ax)
ax.set_title('Distribution des scores CVSS par éditeur (top 8)')
ax.set_xlabel('Éditeur')
ax.set_ylabel('Score CVSS')
ax.tick_params(axis='x', rotation=30)
plt.tight_layout()
plt.savefig('graphique_boxplot_vendors.png')
plt.show()

### 3.9 Évolution temporelle du nombre de CVE publiées

In [ ]:
df_time = df.dropna(subset=['Date de publication']).copy()
df_time['Mois'] = df_time['Date de publication'].dt.to_period('M')
par_mois = df_time.groupby('Mois').size().reset_index(name='Nb CVE')
par_mois['Mois'] = par_mois['Mois'].dt.to_timestamp()

fig, ax = plt.subplots(figsize=(12, 5))
ax.plot(par_mois['Mois'], par_mois['Nb CVE'], marker='o', color='steelblue', linewidth=1.5)
ax.fill_between(par_mois['Mois'], par_mois['Nb CVE'], alpha=0.15, color='steelblue')
ax.set_title('Évolution mensuelle du nombre de CVE publiées')
ax.set_xlabel('Mois')
ax.set_ylabel('Nombre de CVE')
ax.xaxis.set_major_formatter(mdates.DateFormatter('%b %Y'))
ax.xaxis.set_major_locator(mdates.MonthLocator(interval=2))
plt.xticks(rotation=45)
plt.tight_layout()
plt.savefig('graphique_evolution_temporelle.png')
plt.show()

### 3.10 Courbe cumulative des vulnérabilités

In [ ]:
par_mois_cum = par_mois.copy()
par_mois_cum['Cumulé'] = par_mois_cum['Nb CVE'].cumsum()

fig, ax = plt.subplots(figsize=(12, 5))
ax.plot(par_mois_cum['Mois'], par_mois_cum['Cumulé'], color='darkorange', linewidth=2)
ax.fill_between(par_mois_cum['Mois'], par_mois_cum['Cumulé'], alpha=0.15, color='darkorange')
ax.set_title('Courbe cumulative des CVE détectées dans le temps')
ax.set_xlabel('Mois')
ax.set_ylabel('CVE cumulées')
ax.xaxis.set_major_formatter(mdates.DateFormatter('%b %Y'))
ax.xaxis.set_major_locator(mdates.MonthLocator(interval=2))
plt.xticks(rotation=45)
plt.tight_layout()
plt.savefig('graphique_cumul.png')
plt.show()

### 3.11 Avis vs Alertes par éditeur (top 8)

In [ ]:
df_type = df[df['Éditeur (Vendor)'].replace('n/a', pd.NA).notna()].copy()
top8_type = df_type['Éditeur (Vendor)'].value_counts().head(8).index
df_type = df_type[df_type['Éditeur (Vendor)'].isin(top8_type)]
pivot = df_type.groupby(['Éditeur (Vendor)', 'Type']).size().unstack(fill_value=0)

fig, ax = plt.subplots(figsize=(12, 6))
pivot.plot(kind='bar', ax=ax, color=['steelblue','coral'], edgecolor='white')
ax.set_title('Nombre de CVE par éditeur et type de bulletin')
ax.set_xlabel('Éditeur')
ax.set_ylabel('Nombre de CVE')
ax.tick_params(axis='x', rotation=30)
ax.legend(title='Type')
plt.tight_layout()
plt.savefig('graphique_avis_vs_alertes.png')
plt.show()

### 3.12 Analyse par type de faille CWE (top 10)

In [ ]:
df_cwe = df[~df['Type CWE'].isin(['Non disponible','Non renseigné'])].dropna(subset=['Type CWE'])
top_cwe = df_cwe['Type CWE'].value_counts().head(10)

fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Barplot
top_cwe.plot(kind='bar', ax=axes[0], color='teal', edgecolor='white')
axes[0].set_title('Top 10 des types de faille CWE')
axes[0].set_xlabel('CWE')
axes[0].set_ylabel('Nombre de CVE')
axes[0].tick_params(axis='x', rotation=30)

# Score CVSS moyen par CWE
df_cwe_cvss = df_cwe[df_cwe['Type CWE'].isin(top_cwe.index)].dropna(subset=['Score CVSS'])
cwe_cvss_moy = df_cwe_cvss.groupby('Type CWE')['Score CVSS'].mean().reindex(top_cwe.index)
cwe_cvss_moy.plot(kind='bar', ax=axes[1], color='darkcyan', edgecolor='white')
axes[1].set_title('Score CVSS moyen par type CWE (top 10)')
axes[1].set_xlabel('CWE')
axes[1].set_ylabel('Score CVSS moyen')
axes[1].tick_params(axis='x', rotation=30)

plt.tight_layout()
plt.savefig('graphique_cwe.png')
plt.show()

### 3.13 Nombre de CVE par type de bulletin

In [ ]:
type_bulletin = df.groupby('Type')['Identifiant CVE'].count()
fig, ax = plt.subplots(figsize=(7, 5))
type_bulletin.plot(kind='bar', ax=ax, color=['steelblue','coral'], edgecolor='white')
ax.set_title('Nombre de CVE par type de bulletin')
ax.set_xlabel('Type de bulletin')
ax.set_ylabel('Nombre de CVE')
ax.tick_params(axis='x', rotation=0)
plt.tight_layout()
plt.savefig('graphique_type_bulletin.png')
plt.show()

## 4. Machine Learning

Deux modèles sont appliqués aux données enrichies :
- **Non supervisé** : KMeans — regroupement des CVE selon leur profil de risque (CVSS + EPSS)
- **Supervisé** : Random Forest — prédiction du niveau de sévérité à partir des caractéristiques disponibles

### 4.1 Préparation des données ML

In [ ]:
df_ml = df[df['Base Severity'].isin(['LOW','MEDIUM','HIGH','CRITICAL'])].copy()
df_ml = df_ml.dropna(subset=['Score CVSS', 'Score EPSS'])

le_cwe = LabelEncoder()
df_ml['CWE_encoded'] = le_cwe.fit_transform(df_ml['Type CWE'].fillna('Inconnu'))

le_severity = LabelEncoder()
df_ml['Severity_encoded'] = le_severity.fit_transform(df_ml['Base Severity'])

print(f'Lignes utilisées pour le ML : {len(df_ml)}')
print(f'Classes de sévérité : {list(le_severity.classes_)}')

### 4.2 Modèle non supervisé — KMeans

On regroupe les CVE selon leur score CVSS et EPSS sans étiquettes. Le nombre optimal de clusters est déterminé par la méthode du coude (elbow) et le score silhouette.

In [ ]:
X_kmeans = df_ml[['Score CVSS', 'Score EPSS']].copy()
scaler = StandardScaler()
X_scaled = scaler.fit_transform(X_kmeans)

# Méthode du coude
inerties = []
silhouettes = []
K_range = range(2, 9)
for k in K_range:
    km = KMeans(n_clusters=k, random_state=42, n_init=10)
    labels = km.fit_predict(X_scaled)
    inerties.append(km.inertia_)
    silhouettes.append(silhouette_score(X_scaled, labels))

fig, axes = plt.subplots(1, 2, figsize=(14, 5))

axes[0].plot(K_range, inerties, marker='o', color='steelblue')
axes[0].set_title('Méthode du coude (Elbow)')
axes[0].set_xlabel('Nombre de clusters k')
axes[0].set_ylabel('Inertie')

axes[1].plot(K_range, silhouettes, marker='o', color='coral')
axes[1].set_title('Score Silhouette selon k')
axes[1].set_xlabel('Nombre de clusters k')
axes[1].set_ylabel('Score Silhouette')

plt.tight_layout()
plt.savefig('graphique_elbow_silhouette.png')
plt.show()

k_optimal = K_range[silhouettes.index(max(silhouettes))]
print(f'k optimal selon silhouette : {k_optimal}')

In [ ]:
# Entraînement avec k optimal
kmeans_final = KMeans(n_clusters=k_optimal, random_state=42, n_init=10)
df_ml['Cluster'] = kmeans_final.fit_predict(X_scaled)

palette = plt.cm.get_cmap('tab10', k_optimal)
fig, ax = plt.subplots(figsize=(10, 6))
for i in range(k_optimal):
    groupe = df_ml[df_ml['Cluster'] == i]
    ax.scatter(groupe['Score CVSS'], groupe['Score EPSS'],
               label=f'Cluster {i}', color=palette(i), alpha=0.7)
ax.set_title(f'Clustering KMeans (k={k_optimal}) des CVE')
ax.set_xlabel('Score CVSS')
ax.set_ylabel('Score EPSS')
ax.legend()
plt.tight_layout()
plt.savefig('graphique_kmeans.png')
plt.show()

print('Profil moyen par cluster :')
print(df_ml.groupby('Cluster')[['Score CVSS','Score EPSS']].mean().round(3))

### 4.3 Modèle supervisé — Random Forest

On prédit le niveau de sévérité (LOW / MEDIUM / HIGH / CRITICAL) à partir du score CVSS, du score EPSS et du type de faille CWE encodé.

**Justification** : le Random Forest est robuste aux données déséquilibrées, gère bien les features hétérogènes, et fournit un indicateur d'importance des variables facilement interprétable.

In [ ]:
X = df_ml[['Score CVSS', 'Score EPSS', 'CWE_encoded']]
y = df_ml['Severity_encoded']

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y
)

clf = RandomForestClassifier(n_estimators=100, random_state=42, class_weight='balanced')
clf.fit(X_train, y_train)
y_pred = clf.predict(X_test)

print('Rapport de classification :')
print(classification_report(
    y_test, y_pred,
    labels=list(range(len(le_severity.classes_))),
    target_names=le_severity.classes_,
    zero_division=0
))

#### Matrice de confusion

In [ ]:
cm = confusion_matrix(y_test, y_pred)
fig, ax = plt.subplots(figsize=(7, 5))
sns.heatmap(cm, annot=True, fmt='d', cmap='Blues', ax=ax,
            xticklabels=le_severity.classes_,
            yticklabels=le_severity.classes_)
ax.set_title('Matrice de confusion — Random Forest')
ax.set_xlabel('Prédit')
ax.set_ylabel('Réel')
plt.tight_layout()
plt.savefig('graphique_confusion_matrix.png')
plt.show()

#### Importance des variables

In [ ]:
importances = clf.feature_importances_
noms = ['Score CVSS', 'Score EPSS', 'Type CWE']
idx = importances.argsort()[::-1]

fig, ax = plt.subplots(figsize=(8, 5))
ax.bar([noms[i] for i in idx], importances[idx],
       color=['steelblue','coral','mediumseagreen'])
ax.set_title('Importance des variables — Random Forest')
ax.set_ylabel('Importance')
plt.tight_layout()
plt.savefig('graphique_importance_features.png')
plt.show()

## 5. Synthèse

Les visualisations et modèles permettent de dégager plusieurs enseignements :

- Les vulnérabilités **CRITICAL** et **HIGH** représentent la majorité des CVE référencées par l'ANSSI.
- Le score EPSS est peu corrélé au CVSS : une faille peut être très grave (CVSS élevé) sans être activement exploitée, et vice versa.
- Le clustering KMeans identifie des profils de risque distincts, utiles pour prioriser les correctifs.
- Le Random Forest prédit correctement la sévérité avec le CVSS comme variable la plus discriminante.
- Les éditeurs les plus touchés concentrent souvent plusieurs CVE critiques, ce qui justifie une veille ciblée.